# Hero Ranking by Marginal Value (v2 Timing-Aware)

Contrast Monte Carlo: for each hero × map, measure how much better
a player performs *with* the hero vs. blind random bidding.

**Key question**: *Which hero gives the biggest advantage, and on which maps?*

v2 applies timing weights: R1 information is worth 100%, R5 information
is worth ~5%. This correctly penalizes late-info heroes like 拉文.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from bidking_lab.extract.bid_map_table import load_bid_map_table
from bidking_lab.extract.drop_table import load_drop_table
from bidking_lab.extract.item_table import load_item_table
from bidking_lab.simulation.hero_skills import HERO_SKILLS
from bidking_lab.simulation.hero_value import simulate_hero_value

TABLES = ROOT / "data" / "raw" / "tables"
all_maps = load_bid_map_table(TABLES / "BidMap.txt")
drops = load_drop_table(TABLES / "Drop.txt")
items = load_item_table(TABLES / "Item.txt")

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print(f"Loaded {len(all_maps)} maps, {len(HERO_SKILLS)} heroes")

## 1. Run contrast MC on 5 representative maps

In [ ]:
MAP_IDS = [2107, 2205, 2301, 2407, 2505]
MAP_SHORT = {2107: "快递", 2205: "仓库", 2301: "集装箱", 2407: "别墅", 2505: "沉船"}

N_TRIALS = 10_000
rng = np.random.default_rng(20260515)
hero_ids = sorted(HERO_SKILLS.keys())

results = []
for mid in MAP_IDS:
    for hid in hero_ids:
        r = simulate_hero_value(
            hid, mid, maps=all_maps, drops=drops, items=items,
            n_trials=N_TRIALS, use_timing=True, rng=rng,
        )
        results.append(r)

print(f"Ran {len(results)} hero × map combinations")

## 2. Heatmap — marginal value %

In [ ]:
import pandas as pd

rows = []
for r in results:
    rows.append({
        "hero": f"{HERO_SKILLS[r.hero_id].name}",
        "hero_id": r.hero_id,
        "map": MAP_SHORT.get(r.map_id, r.map_name),
        "marginal_pct": r.marginal_pct,
        "marginal_value": r.marginal_value,
    })
df = pd.DataFrame(rows)

avg_rank = (
    df.groupby("hero")["marginal_pct"]
    .mean()
    .sort_values(ascending=False)
)
hero_order = avg_rank.index.tolist()
map_order = [MAP_SHORT[m] for m in MAP_IDS]

pivot = df.pivot(index="hero", columns="map", values="marginal_pct")
pivot = pivot.reindex(index=hero_order, columns=map_order)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    pivot, annot=True, fmt="+.1f", cmap="RdYlGn",
    center=0, linewidths=0.5, ax=ax,
    cbar_kws={"label": "Marginal Value %"},
)
ax.set_title("Hero Marginal Value % (v2 timing-aware)", fontsize=13)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(str(ROOT / "notebooks" / "fig_hero_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Top-5 heroes bar chart per map

In [ ]:
fig, axes = plt.subplots(1, len(MAP_IDS), figsize=(18, 5), sharey=False)

for ax, mid in zip(axes, MAP_IDS):
    sub = df[df["map"] == MAP_SHORT[mid]].nlargest(5, "marginal_pct")
    colors = sns.color_palette("viridis", n_colors=5)
    ax.barh(
        sub["hero"].values[::-1],
        sub["marginal_pct"].values[::-1],
        color=colors[::-1],
    )
    ax.set_title(MAP_SHORT[mid])
    ax.set_xlabel("Marginal %")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Top 5 Heroes by Map (marginal value %)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(ROOT / "notebooks" / "fig_hero_top5.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. v1 vs v2 comparison — Raven correction

In [ ]:
rng2 = np.random.default_rng(20260515)
compare_mid = 2407

v1_results = []
v2_results = []
for hid in hero_ids:
    r_v1 = simulate_hero_value(
        hid, compare_mid, maps=all_maps, drops=drops, items=items,
        n_trials=N_TRIALS, use_timing=False, rng=rng2,
    )
    v1_results.append(r_v1)

rng3 = np.random.default_rng(20260515)
for hid in hero_ids:
    r_v2 = simulate_hero_value(
        hid, compare_mid, maps=all_maps, drops=drops, items=items,
        n_trials=N_TRIALS, use_timing=True, rng=rng3,
    )
    v2_results.append(r_v2)

cmp_rows = []
for r1, r2 in zip(v1_results, v2_results):
    cmp_rows.append({
        "hero": HERO_SKILLS[r1.hero_id].name,
        "v1_pct": r1.marginal_pct,
        "v2_pct": r2.marginal_pct,
        "delta": r2.marginal_pct - r1.marginal_pct,
    })
cdf = pd.DataFrame(cmp_rows).sort_values("delta")

fig, ax = plt.subplots(figsize=(10, 8))
y = range(len(cdf))
ax.barh(y, cdf["v1_pct"].values, height=0.35, label="v1 (no timing)", color="#aaa", align="edge")
ax.barh([i - 0.35 for i in y], cdf["v2_pct"].values, height=0.35, label="v2 (timing-aware)", color="#4c72b0", align="edge")
ax.set_yticks(y)
ax.set_yticklabels(cdf["hero"].values)
ax.set_xlabel("Marginal Value %")
ax.set_title(f"v1 vs v2 Hero Rankings — {all_maps[compare_mid].name}", fontsize=13)
ax.legend()
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(str(ROOT / "notebooks" / "fig_v1_vs_v2.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nBiggest rank changes (v1 → v2):")
print(cdf[["hero", "v1_pct", "v2_pct", "delta"]].to_string(index=False))

## Takeaways

1. **玛丽亚 (Maria)** is the best overall hero — R1 value info on white/green/blue items covers 70%+ of the pool.
2. **拉文 (Raven)** drops from top-tier in v1 to near-zero in v2 — R5 quality info is too late to help.
3. **艾哈迈德 (Ahmed)** also drops — COUNT_HINT (statistical aggregates) doesn't help per-item selection.
4. **索菲 (Sophie)** and **加布里埃拉 (Gabriella)** are strong generalists with progressive quality reveals.
5. Category-specialist heroes (乔治/weapons, 伊万/weapons+energy) shine on maps heavy in their category.